# Entity Extraction: Rule-Based + spaCy NER

This notebook demonstrates two of the three approaches from `../05-entity-extraction-and-ner.md` for
"Entity-extraction-based Project demand capture": **rule-based regex extraction** and **spaCy
statistical NER**. (The third approach, LLM-based structured extraction with Pydantic, is shown as
code in the chapter itself -- it needs a real LLM API call, so it isn't reproduced as a runnable cell
here; this notebook stays fully offline.)

The spaCy cell is wrapped in a `try/except` so the notebook **runs to completion even if spaCy or its
`en_core_web_sm` model isn't installed** -- it prints the expected output shape instead of failing.

In [1]:
import re

sample_request = (
    "Hi team, for Project Atlas we need the promotional deck localized into Japanese and Korean "
    "for the APAC region, due by end of quarter. Similar to what we did last time."
)
print(sample_request)

Hi team, for Project Atlas we need the promotional deck localized into Japanese and Korean for the APAC region, due by end of quarter. Similar to what we did last time.


## 1. Rule-based / regex extraction

Hand-written patterns for the fields the project-demand-capture feature cares about: project name,
languages, region, and deadline. Fast, deterministic, zero dependencies -- but only catches phrasing
it was explicitly written to catch (see Chapter 5's discussion of this trade-off).

In [2]:
PROJECT_PATTERN = re.compile(r"\bProject\s+([A-Z][a-zA-Z]+)\b")
LANGUAGE_PATTERN = re.compile(
    r"\b(Japanese|Korean|French|German|Spanish|Mandarin|Portuguese)\b", re.IGNORECASE
)
REGION_PATTERN = re.compile(r"\b(APAC|EMEA|EU|NA|LATAM)\b")
DEADLINE_PATTERN = re.compile(
    r"\b(?:by|before|due by|due)\s+([A-Za-z]+\s+\d{1,2}(?:,?\s+\d{4})?|end of \w+|Q[1-4]\s*\d{0,4})",
    re.IGNORECASE,
)


def rule_based_extract(text: str) -> dict:
    project_match = PROJECT_PATTERN.search(text)
    deadline_match = DEADLINE_PATTERN.search(text)
    return {
        "project_name": f"Project {project_match.group(1)}" if project_match else None,
        "languages": LANGUAGE_PATTERN.findall(text),
        "region": REGION_PATTERN.findall(text),
        "deadline": deadline_match.group(1) if deadline_match else None,
    }


rule_result = rule_based_extract(sample_request)
rule_result

{'project_name': 'Project Atlas',
 'languages': ['Japanese', 'Korean'],
 'region': ['APAC'],
 'deadline': 'end of quarter'}

## 2. Where rule-based extraction breaks

Rephrase the same request slightly -- no exact keyword match for the deadline pattern, and the
project reference uses a different phrasing. This is the concrete version of Chapter 5's point:
regex only catches phrasing it was written to catch.

In [3]:
rephrased_request = (
    "We'll need JP and KR versions of the deck for the Atlas engagement, ideally wrapped up by "
    "the end of the quarter, same as the previous rollout."
)

rephrased_result = rule_based_extract(rephrased_request)
print("Rephrased extraction result:", rephrased_result)
print()
print("Note: 'project_name' is None (pattern requires the literal words 'Project <Name>'), and")
print("'languages' is empty (pattern requires 'Japanese'/'Korean' spelled out, not 'JP'/'KR').")
print("This is exactly the brittleness Chapter 5 flags as rule-based extraction's core weakness.")

Rephrased extraction result: {'project_name': None, 'languages': [], 'region': [], 'deadline': None}

Note: 'project_name' is None (pattern requires the literal words 'Project <Name>'), and
'languages' is empty (pattern requires 'Japanese'/'Korean' spelled out, not 'JP'/'KR').
This is exactly the brittleness Chapter 5 flags as rule-based extraction's core weakness.


## 3. spaCy statistical NER (with graceful fallback)

spaCy's pretrained pipeline recognizes general-purpose entity types (`GPE`, `DATE`, `NORP`, `ORG`)
using a trained model rather than hand-written patterns, so it generalizes better to phrasing
variations -- but has no built-in concept of "project name" as a domain-specific field without
further customization (an `EntityRuler` or fine-tuning, per Chapter 5).

This cell is wrapped in `try/except` so the notebook completes even without spaCy installed.

In [4]:
try:
    import spacy

    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        raise ImportError(
            "spaCy is installed but the 'en_core_web_sm' model isn't downloaded "
            "(run: python -m spacy download en_core_web_sm)"
        )

    doc = nlp(sample_request)
    spacy_entities = [(ent.text, ent.label_) for ent in doc.ents]

    print("spaCy NER result:")
    for text, label in spacy_entities:
        print(f"  {text!r:30s} -> {label}")

except (ImportError, ModuleNotFoundError) as e:
    print(f"[spaCy unavailable: {e}]")
    print()
    print("Falling back to the EXPECTED OUTPUT SHAPE spaCy would produce on this sentence,")
    print("so the notebook still demonstrates what statistical NER output looks like:\n")

    expected_spacy_output = [
        ("Atlas", "ORG"),          # project codenames often get misclassified as ORG/PRODUCT
        ("Japanese", "NORP"),      # nationalities/religious/political groups -- languages land here
        ("Korean", "NORP"),
        ("APAC", "ORG"),           # region acronyms often get misclassified without a custom ruler
    ]
    for text, label in expected_spacy_output:
        print(f"  {text!r:30s} -> {label}")
    print()
    print("Note the mismatch with what we actually want (project_name, language, region, deadline")
    print("as clean domain-specific fields) -- this is exactly why Chapter 5 argues a custom")
    print("EntityRuler or fine-tuning (or LLM-based structured extraction) is needed on top of the")
    print("out-of-the-box spaCy model for this use case.")

[spaCy unavailable: No module named 'spacy']

Falling back to the EXPECTED OUTPUT SHAPE spaCy would produce on this sentence,
so the notebook still demonstrates what statistical NER output looks like:

  'Atlas'                        -> ORG
  'Japanese'                     -> NORP
  'Korean'                       -> NORP
  'APAC'                         -> ORG

Note the mismatch with what we actually want (project_name, language, region, deadline
as clean domain-specific fields) -- this is exactly why Chapter 5 argues a custom
EntityRuler or fine-tuning (or LLM-based structured extraction) is needed on top of the
out-of-the-box spaCy model for this use case.


## 4. Why this motivates LLM-based structured extraction (Chapter 5, Section 3)

Neither approach above cleanly produces the target schema (`asset_type`, `languages`, `region`,
`deadline`, `reference_project`) on both phrasings of the request without extra engineering: the
rule-based extractor fails on `rephrased_request` entirely, and spaCy's out-of-the-box labels
(`ORG`, `NORP`) don't map directly onto our domain fields without a custom `EntityRuler` or
fine-tuning. `../05-entity-extraction-and-ner.md` walks through the LLM-based structured-output
approach (a `Pydantic` schema + `PydanticOutputParser` over a `ChatPromptTemplate`) that handles both
phrasings in one pass without any of that extra engineering -- it isn't reproduced as a runnable cell
here since it requires a real LLM API call, but the code is shown in full in the chapter.

## Takeaways

- Rule-based extraction is fast and dependency-free but brittle to rephrasing -- see the
  `rephrased_request` failure above.
- spaCy's statistical NER generalizes better than regex but needs domain customization
  (`EntityRuler` / fine-tuning) to produce clean, task-specific fields rather than generic entity
  types.
- Both approaches motivate the LLM-based structured-extraction approach used for the real "project
  demand capture" feature (Chapter 5) -- one schema, works across phrasings and languages, no
  training data required, at the cost of an LLM API call per extraction.